# Idea 1 (aligned count) replayed on the pipeline's REAL candidates, four photo sets, thresholds fixed on two and tested on the other two
Idea 1b injected the true place as a candidate, so it could not show wrong answers. Here no truth is injected: the dumps `lowconf_<set>` hold, for the 6 standard queries of every photo, the 10 fine-matched candidates with LoFTR run at a low coarse threshold (0.05) and matches kept down to 0.05. 
Feature of a candidate: matches (confidence >= C) whose two points lie within T metres of the same pixel, minus the count a random scatter gives, over its square root (z). Rule (rescue): a photo that today ends in SKIP (after the 27-query escalation, from the reference dump `<set>`) is answered at the candidate with the highest z if that z >= thr and it beats the runner-up place by a margin.
Cost: none beyond the six queries already run (a count over stored matches); the one cost is that LoFTR runs at a lower coarse threshold, measured by the dump's own `fine` seconds against the reference.
Sets: DEV = official150 + nudt_valid1212 (choose C, T, thr, margin here); TEST = nudt_fresh707 + nudt_fresh1010 (never used to choose anything).
Falsification (planned): dead if on TEST the rule gives any wrong answer (-20) or gains less than 1 point of the /60 score; also dead if the low coarse threshold changes the answered photos of today's rule (stored inliers must agree with the reference).

In [1]:
import os, sys, math
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"
from train.checks import replay
from fine_matching.loftr_matcher import reliable_mask
from image_processing.map_levels import level_gsd
SETS = ["official150", "nudt_valid1212", "nudt_fresh707", "nudt_fresh1010"]
DEV, TEST = SETS[:2], SETS[2:]
ref = {s: replay.load_set(T7 / "rich_dump" / s) for s in SETS}
low = {s: replay.load_set(T7 / "rich_dump" / ("lowconf_" + s), with_matches=True) for s in SETS if (T7 / "rich_dump" / ("lowconf_" + s)).exists()}
print({s: (len(ref[s]), len(low.get(s, {}))) for s in SETS})

{'official150': (150, 150), 'nudt_valid1212': (150, 150), 'nudt_fresh707': (150, 150), 'nudt_fresh1010': (133, 133)}


In [2]:
# check 1: the low threshold must not change today's decisions (stored inliers read the matches at 0.22)
same = diff = 0
for s in low:
    for pid, photo in low[s].items():
        if pid not in ref[s]:
            continue
        a = replay.decide_current(photo, escalate=False); b = replay.decide_current(ref[s][pid], escalate=False)
        same += int(a["stage"] == b["stage"] and abs(a["err_sigma"] - b["err_sigma"]) < 1e-6); diff += int(not (a["stage"] == b["stage"] and abs(a["err_sigma"] - b["err_sigma"]) < 1e-6))
print("first-pass decisions identical to the reference dump: %d, different: %d" % (same, diff))
sec = lambda d: np.mean([sum(sum(q["seconds"].get(k, 0) for k in ("fine",) ) for q in photo["queries"]) for photo in d.values()])
for s in low:
    print(s, "fine seconds per photo: low-threshold %.2f s (6 queries) vs reference first 6 queries %.2f s" % (sec(low[s]), np.mean([sum(photo["queries"][i]["seconds"].get("fine", 0) for i in replay.query_ids(photo, replay.STANDARD)) for photo in ref[s].values()])))

first-pass decisions identical to the reference dump: 580, different: 3
official150 fine seconds per photo: low-threshold 2.32 s (6 queries) vs reference first 6 queries 2.39 s
nudt_valid1212 fine seconds per photo: low-threshold 2.37 s (6 queries) vs reference first 6 queries 2.32 s
nudt_fresh707 fine seconds per photo: low-threshold 2.29 s (6 queries) vs reference first 6 queries 2.36 s
nudt_fresh1010 fine seconds per photo: low-threshold 2.26 s (6 queries) vs reference first 6 queries 2.39 s


In [3]:
SIZE = 224
def z_of(photo, level, c, t):
    """For every standard query and candidate: (z, x, y, inliers) with z the excess count of aligned matches over chance."""
    gsd = level_gsd(level); out = []
    for qi in replay.query_ids(photo, replay.STANDARD):
        q = photo["queries"][qi]; reliable = reliable_mask(q["inside"], q["border_px"])
        for k, m in enumerate(q["matches"]):
            if m is None or q["peaks"][k][1] > 1e8:
                continue
            a, b, conf = m["points_a"], m["points_b"], m["confidence"]
            cols = np.clip(np.round(a[:, 0]).astype(int), 0, reliable.shape[1] - 1); rows = np.clip(np.round(a[:, 1]).astype(int), 0, reliable.shape[0] - 1)
            chosen = reliable[rows, cols] & (conf >= c)
            n = int(chosen.sum()); gap = np.hypot(b[chosen, 0] - a[chosen, 0], b[chosen, 1] - a[chosen, 1]) * gsd
            near = int((gap <= t).sum()); chance = n * min(1.0, math.pi * (t / gsd) ** 2 / (SIZE * SIZE))
            out.append(((near - chance) / math.sqrt(chance + 1.0), q["peaks"][k][1] + q["shifts"][k][0], q["peaks"][k][2] + q["shifts"][k][1], float(q["inliers"][k]), qi))
    return out
C_LIST, T_LIST = [0.05, 0.10, 0.15], [8.0, 12.0, 20.0]
cache = {}
for s in low:
    for pid, photo in low[s].items():
        for c in C_LIST:
            for t in T_LIST:
                cache[(s, pid, c, t)] = z_of(photo, photo["level"], c, t)
print(len(cache), "photo x setting entries")

5247 photo x setting entries


In [4]:
def rescue(cands, photo, thr, margin, radius_sigma=2.0):
    """Answer at the candidate with the highest z if z >= thr and no candidate farther than radius_sigma from it has z within `margin` of it."""
    if not cands:
        return None
    z = np.array([c[0] for c in cands]); best = int(np.argmax(z))
    if z[best] < thr:
        return None
    far = [c[0] for c in cands if math.hypot(c[1] - cands[best][1], c[2] - cands[best][2]) / photo["sigma"] > radius_sigma]
    if far and max(far) > z[best] - margin:
        return None
    return cands[best][1], cands[best][2]

def evaluate(s, c, t, thr, margin, mode):
    """mode 'after_escalation': today's full rule, then rescue on photos still skipped. 'instead_of_escalation': rescue right after a skipped first pass, then escalate if still no answer."""
    rows = []
    for pid, photo in ref[s].items():
        if pid not in low[s]:
            continue
        today = replay.decide_current(photo); first = replay.decide_current(photo, escalate=False)
        points, wrong, got, used = today["points"], int(today["answered"] and today["err_sigma"] > 8), 0, today["n_queries"]
        base_points, base_wrong = points, wrong
        trigger = (today["stage"] == "skip") if mode == "after_escalation" else (first["stage"] == "skip")
        if trigger:
            r = rescue(cache[(s, pid, c, t)], photo, thr, margin)
            if r is not None:
                err = replay.error_sigma(photo, *r); points, wrong, got = replay.points(err), int(err > 8), 1
                used = 6 if mode == "instead_of_escalation" else today["n_queries"]
        rows.append(dict(set=s, photo=pid, base_points=base_points, base_wrong=base_wrong, points=points, wrong=wrong, rescued=got, queries=used, base_queries=today["n_queries"]))
    return pd.DataFrame(rows)

def summarize(df):
    n = len(df)
    return dict(photos=n, base_score=round(df.base_points.sum() / n * 6 - 20 * 0, 2), score=round(df.points.sum() / n * 6, 2), rescued=int(df.rescued.sum()),
                wrong_base=int(df.base_wrong.sum()), wrong=int(df.wrong.sum()), queries_saved_per_photo=round((df.base_queries - df.queries).mean(), 2))

In [5]:
# the grid is chosen on DEV only: the best total score on the two dev sets with no extra wrong answer on either; all cells are printed so a lucky cell is visible
grid = []
THR = [2, 3, 4, 5, 6, 8, 10, 14]; MARGIN = [0, 1, 2, 4]
for c in C_LIST:
    for t in T_LIST:
        for thr in THR:
            for margin in MARGIN:
                for mode in ["after_escalation"]:
                    dev = pd.concat([evaluate(s, c, t, thr, margin, mode) for s in DEV if s in low])
                    row = dict(C=c, T=t, thr=thr, margin=margin, mode=mode, **{k: v for k, v in summarize(dev).items() if k in ("rescued", "wrong", "wrong_base", "base_score", "score")})
                    grid.append(row)
G = pd.DataFrame(grid); G["gain"] = G.score - G.base_score; G["extra_wrong"] = G.wrong - G.wrong_base
G.to_csv(T7 / "idea1c_dev_grid.csv", index=False)
ok = G[G.extra_wrong <= 0].sort_values(["gain", "rescued"], ascending=False)
print("DEV grid cells with no extra wrong answer: %d of %d" % (len(ok), len(G)))
print(ok.head(15).to_string(index=False))

DEV grid cells with no extra wrong answer: 38 of 288
   C    T  thr  margin             mode  base_score  score  rescued  wrong_base  wrong  gain  extra_wrong
0.05 20.0   10       4 after_escalation        52.2  55.24       16           3      3  3.04            0
0.05 20.0   14       4 after_escalation        52.2  55.04       15           3      3  2.84            0
0.10 20.0    2       4 after_escalation        52.2  54.64       13           3      3  2.44            0
0.10 20.0    3       4 after_escalation        52.2  54.64       13           3      3  2.44            0
0.10 20.0    4       4 after_escalation        52.2  54.64       13           3      3  2.44            0
0.10 20.0    5       4 after_escalation        52.2  54.64       13           3      3  2.44            0
0.10 20.0    6       4 after_escalation        52.2  54.64       13           3      3  2.44            0
0.10 20.0    8       4 after_escalation        52.2  54.04       10           3      3  1.84       

In [6]:
# TEST: the single cell picked on DEV, applied unchanged; and the neighbourhood so a knife-edge pick shows
best = ok.iloc[0]
print("picked on DEV:", dict(best[["C", "T", "thr", "margin"]]))
res = []
for mode in ["after_escalation", "instead_of_escalation"]:
    for s in SETS:
        if s in low:
            df = evaluate(s, best["C"], best["T"], best["thr"], best["margin"], mode)
            res.append(dict(mode=mode, set=s, role="DEV" if s in DEV else "TEST", **summarize(df)))
R = pd.DataFrame(res); R.to_csv(T7 / "idea1c_result.csv", index=False); print(R.to_string(index=False))
near = G[(G["C"] == best["C"]) & (G["T"] == best["T"])].pivot(index="thr", columns="margin", values="gain")
print("DEV gain around the pick (rows thr, columns margin):"); print(near.to_string())
wrongs = G[(G["C"] == best["C"]) & (G["T"] == best["T"])].pivot(index="thr", columns="margin", values="extra_wrong")
print("extra wrong answers around the pick:"); print(wrongs.to_string())

picked on DEV: {'C': 0.05, 'T': 20.0, 'thr': 10, 'margin': 4}


                 mode            set role  photos  base_score  score  rescued  wrong_base  wrong  queries_saved_per_photo
     after_escalation    official150  DEV     150       52.80  58.08       14           0      0                     0.00
     after_escalation nudt_valid1212  DEV     150       51.60  52.40        2           3      3                     0.00
     after_escalation  nudt_fresh707 TEST     150       53.84  53.12        6           1      3                     0.00
     after_escalation nudt_fresh1010 TEST     133       54.09  52.29        3           1      3                     0.00
instead_of_escalation    official150  DEV     150       52.80  58.08       15           0      0                     2.10
instead_of_escalation nudt_valid1212  DEV     150       51.60  52.40        2           3      3                     0.28
instead_of_escalation  nudt_fresh707 TEST     150       53.84  52.96        7           1      3                     0.98
instead_of_escalation nu

## Exploratory only (the TEST sets are seen here, so nothing in this table counts as a test)
How strict must the rule be before the held-out NUDT sets stop getting extra wrong answers, and what is left of the official gain then?

In [7]:
rows = []
for c, t in [(0.05, 20.0), (0.10, 20.0), (0.15, 12.0)]:
    for thr in [10, 14, 20, 30, 45]:
        for margin in [4, 8, 16]:
            line = dict(C=c, T=t, thr=thr, margin=margin)
            for s in SETS:
                df = evaluate(s, c, t, thr, margin, "after_escalation")
                line[s] = "%+.2f / %+dw / %dr" % (df.points.sum() / len(df) * 6 - df.base_points.sum() / len(df) * 6, int(df.wrong.sum() - df.base_wrong.sum()), int(df.rescued.sum()))
            rows.append(line)
S = pd.DataFrame(rows); S.to_csv(T7 / "idea1c_strict_sweep.csv", index=False)
print("cell = score gain of 60 / extra wrong answers / photos rescued"); print(S.to_string(index=False))

cell = score gain of 60 / extra wrong answers / photos rescued
   C    T  thr  margin       official150   nudt_valid1212    nudt_fresh707   nudt_fresh1010
0.05 20.0   10       4 +5.28 / +0w / 14r +0.80 / +0w / 2r -0.72 / +2w / 6r -1.80 / +2w / 3r
0.05 20.0   10       8 +3.68 / +0w / 10r +0.40 / +0w / 1r +0.48 / +0w / 2r +0.00 / +0w / 0r
0.05 20.0   10      16  +1.04 / +0w / 3r +0.40 / +0w / 1r +0.24 / +0w / 1r +0.00 / +0w / 0r
0.05 20.0   14       4 +4.88 / +0w / 13r +0.80 / +0w / 2r -1.12 / +2w / 5r -0.90 / +1w / 2r
0.05 20.0   14       8 +3.68 / +0w / 10r +0.40 / +0w / 1r +0.48 / +0w / 2r +0.00 / +0w / 0r
0.05 20.0   14      16  +1.04 / +0w / 3r +0.40 / +0w / 1r +0.24 / +0w / 1r +0.00 / +0w / 0r
0.05 20.0   20       4  +1.44 / +0w / 4r +0.40 / +0w / 1r +0.48 / +0w / 2r +0.00 / +0w / 0r
0.05 20.0   20       8  +1.44 / +0w / 4r +0.40 / +0w / 1r +0.48 / +0w / 2r +0.00 / +0w / 0r
0.05 20.0   20      16  +1.04 / +0w / 3r +0.40 / +0w / 1r +0.24 / +0w / 1r +0.00 / +0w / 0r
0.05 20.0   30   